# Stage 2 — Shared Lakehouse Setup

Personal setup log for working against the shared Azure/Databricks infrastructure: container, external location, medallion schemas, and secret scope.

## 1. Personal container

Created container `ivanrazumovskyi` in the shared ADLS account `dlspl21databricks`

In [0]:
# Verify the personal container is reachable
display(dbutils.fs.ls("abfss://ivanrazumovskyi@dlspl21databricks.dfs.core.windows.net/"))

## 2. Unity Catalog external location

Created external location, pointing at the personal container and using the shared storage credential (`databricks_uc_connector`).

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `ivanrazumovskyi_external_location`;

## 3. Medallion schemas

Created three schemas in the shared catalog `dbr_dev`

* `dbr_dev.ivanrazumovskyi_bronze` → `.../schemas/bronze/`
* `dbr_dev.ivanrazumovskyi_silver` → `.../schemas/silver/`
* `dbr_dev.ivanrazumovskyi_gold` → `.../schemas/gold/`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbr_dev.ivanrazumovskyi_bronze
MANAGED LOCATION 'abfss://ivanrazumovskyi@dlspl21databricks.dfs.core.windows.net/schemas/bronze/'
COMMENT 'Bronze schema owned by ivanrazumovskyi';

CREATE SCHEMA IF NOT EXISTS dbr_dev.ivanrazumovskyi_silver
MANAGED LOCATION 'abfss://ivanrazumovskyi@dlspl21databricks.dfs.core.windows.net/schemas/silver/'
COMMENT 'Silver schema owned by ivanrazumovskyi';

CREATE SCHEMA IF NOT EXISTS dbr_dev.ivanrazumovskyi_gold
MANAGED LOCATION 'abfss://ivanrazumovskyi@dlspl21databricks.dfs.core.windows.net/schemas/gold/'
COMMENT 'Gold schema owned by ivanrazumovskyi';

SHOW SCHEMAS IN dbr_dev LIKE 'ivanrazumovskyi*';

## 4. Personal secret scope

Created scope `ivanrazumovskyi-kv-scope`, backed by the shared Key Vault `kvpl24databricks2`.


In [0]:
display(dbutils.secrets.listScopes())

## 5. Job on shared all-purpose cluster

Created Job `ivanrazumovskyi-bronze-ingestion`, running notebook `stage2_bronze_ingestion_job` on the shared all-purpose cluster `GP1`. Successfully run twice.